# Módulo 9 — Recalibración continua

El Módulo 8 dejó el hallazgo operativo más serio del repositorio: el paquete desplegable promete 1% de alertas y entrega **11,10% el día 16**. Las piezas para arreglarlo ya existían —el p-valor conforme del Módulo 6 y la ventana que se refresca del Módulo 7— pero nunca se habían juntado.

Este notebook deja que la calibración avance con el tráfico y compara cuatro estrategias sobre el mismo detector:

| Estrategia | Qué se agrega a la ventana al cerrar el día | Desplegable |
|---|---|---|
| `estatico` | Nada (el paquete del Módulo 8) | Sí |
| `ventana` | Todos los scores del día, sin etiquetas | Sí |
| `ventana_sin_alertas` | Solo lo que no se alertó | Sí |
| `ventana_oraculo` | Solo las legítimas, con etiquetas al instante | No — ablación |

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import RobustScaler

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.operations.temporal import get_temporal_data, period_index
from src.serving.run_recalibration import (
    ALPHA,
    STRATEGIES,
    monitor_agreement,
    plot_recalibration,
    simulate,
    summarize,
)
from src.unsupervised.families import GMMDensity
from src.unsupervised.models import anomaly_score

## 1. Mismo detector que el paquete del Módulo 8

Los scores del período de prueba se calculan **una sola vez**. El detector no cambia entre estrategias; solo cambia la calibración contra la que se comparan esos scores, así que cualquier diferencia en las alertas viene de la calibración y no del modelo.

In [ ]:
data = get_temporal_data()

scaler = RobustScaler().fit(data["X_train"])
detector = GMMDensity().fit(scaler.transform(data["X_train"]))

calibracion = anomaly_score(detector, scaler.transform(data["X_calib"]))
scores = anomaly_score(detector, scaler.transform(data["X_test"]))
periodos = period_index(data["steps_test"], data["cutoff_step"])

print(f"calibración inicial: {calibracion.size:,} | prueba: {scores.size:,} en {len(np.unique(periodos))} días")

## 2. Simulación día por día

Dos decisiones de diseño que no son obvias:

- **La ventana se mide en transacciones (30.000), no en días.** El volumen de PaySim cae 2.111x a fin de mes; una ventana por días terminaría con unos pocos cientos de scores.
- **Primero se puntúa el día y después se lo agrega a la ventana.** Al revés, cada día se calibraría con sus propios scores y la tasa de alertas saldría perfecta por construcción.

In [ ]:
tabla = simulate(calibracion, scores, data["y_test"], periodos)

evaluables = tabla[tabla["n"] >= 1_000]
tasas = evaluables.pivot(index="dia", columns="estrategia", values="tasa_alerta") * 100
tasas[list(STRATEGIES)].round(2)

In [ ]:
resumen = summarize(tabla)
resumen["alertas_por_fraude"] = resumen["alertas"] / (resumen["fraude_capturado"] * int(data["y_test"].sum()))
resumen

In [ ]:
fig = plot_recalibration(tabla, output_path=None)
plt.show()

## 3. ¿La ventana pierde recall por contaminación?

La ventana atrapa 7,2 puntos menos de fraude que el paquete estático. La explicación natural es que el fraude sin etiquetar engorda la cola de la calibración y sube el umbral de más.

La ventana oráculo mide eso: agrega solo legítimas. Si la contaminación explicara la pérdida, recuperaría el recall del estático.

In [ ]:
capturado = resumen.set_index("estrategia")["fraude_capturado"] * 100
perdida = capturado["estatico"] - capturado["ventana"]
recuperado = capturado["ventana_oraculo"] - capturado["ventana"]

print(f"pérdida de recall de la ventana: {perdida:.1f} puntos")
print(f"recuperado sin fraude en la ventana: {recuperado:.1f} puntos ({recuperado / perdida:.0%})")

Recupera alrededor del 18%. La contaminación existe, pero no es la causa principal: el paquete estático atrapa más fraude **porque alerta más** —el 2,77% del tráfico en vez del 1% prometido—. La caída de recall es el costo de cumplir lo prometido, no un defecto de recalibrar.

## 4. Un monitor de deriva sin etiquetas

La FPR real necesita saber qué transacciones eran legítimas, y esas etiquetas llegan semanas tarde. La razón de cola se calcula el mismo día.

In [ ]:
monitor_agreement(tabla)

## 5. Conclusiones

- **Recalibrar devuelve la tasa de alertas a lo prometido.** La FPR media baja de 2,26% a 1,10% y la tasa máxima de 11,09% a 2,32%.
- **Excluir las alertas de la ventana cierra un lazo** que las pruebas predecían y PaySim confirma: el umbral cae de 14,01 a 0,98 y se termina alertando el 48% del tráfico.
- **El recall extra del paquete estático estaba comprado con su exceso de alertas.** La contaminación por fraude explica menos de una quinta parte de la diferencia.
- **La FPR casi perfecta de la ventana es en parte dos sesgos que se cancelan**: el retraso de un día baja el umbral, el fraude lo sube.
- **El mejor monitor de deriva disponible es la propia tasa de alertas comparada con α.** Ordena los días igual que la FPR real (Spearman 0,78) y está disponible el mismo día.